In [10]:
import pandas as pd 
import numpy as np  

import os

import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf

from sklearn.preprocessing import Normalizer, StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers, losses
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.callbacks import ModelCheckpoint
from keras.utils import to_categorical

In [11]:
data = pd.read_hdf("../initialSingleCellDf-channel-20220916-MW_018-001.h5", key="df")
MARKERS = data.columns
# ANTIGENS = ['V4', 'T4', 'Q4', 'A2', 'N4']  # Only focus on the strong markers. 
ANTIGENS = ['null', 'E1', 'G4', 'V4', 'T4', 'Q4', 'A2', 'N4']
TIMES = [4.0, 12.0, 24.0, 30.0, 36.0, 48.0, 60.0, 72.0]
TEST_PROPORTION = 0.25

In [14]:
selected_columns = ['FSC-A', 'SSC-A', 'CD25', 'CD38', 'Granzyme B', 'CD2', 'CD27', 'CD45RA',
       'CD4', 'CD86', 'CXCR6', 'CD5', 'CD62L', 'OX40', 'PD-L1', 'TBet',
       'CD126', 'Proliferation', 'ICOS', 'IRF8', 'CD19', 'MHC-II', 'CD45',
       'CD44', 'CX3CR1', 'CD8a']

# Extract the selected columns into a new DataFrame
# selected_df = data[selected_columns].reset_index(drop=True)
# print(selected_df.columns)
# selected_df

data

Marker                                               FSC-A  SSC-A  CD25  CD38  \
CellType Peptide Concentration Replicate Time Event                             
APCs     N4      1uM           1         4.0  1        771    143   314   259   
                                              2        345     49   260   278   
                                              3        387     82   355   199   
                                              4        379    120   277   241   
                                              5        313    174   313   253   
...                                                    ...    ...   ...   ...   
OT-1     null    10pM          2         72.0 8464     393    184   271   280   
                                              8465     316    108   286   327   
                                              8466     274    131   301   260   
                                              8467     357     91   295   295   
                                              8468     182    129   319   262   

Marker                                               Granzyme B  CD2  CD27  \
CellType Peptide Concentration Replicate Time Event                          
APCs     N4      1uM           1         4.0  1             445  209   337   
                                              2             395  239   273   
                                              3             512  192   298   
                                              4             411  282   274   
                                              5             414  254   280   
...                                                         ...  ...   ...   
OT-1     null    10pM          2         72.0 8464          287  212   288   
                                              8465          284   17   304   
                                              8466          315  447   279   
                                              8467          294  380   268   
                                              8468          278  426   302   

Marker                                               CD45RA  CD4  CD86  ...  \
CellType Peptide Concentration Replicate Time Event                     ...   
APCs     N4      1uM           1         4.0  1         482  227   277  ...   
                                              2         414  261   270  ...   
                                              3         369  225   292  ...   
                                              4         401  247   292  ...   
                                              5         336  260   290  ...   
...                                                     ...  ...   ...  ...   
OT-1     null    10pM          2         72.0 8464      298  246   294  ...   
                                              8465      379  260   304  ...   
                                              8466      278  270   315  ...   
                                              8467      263  253   279  ...   
                                              8468      265  289   277  ...   

Marker                                               CD126  Proliferation  \
CellType Peptide Concentration Replicate Time Event                         
APCs     N4      1uM           1         4.0  1        312            865   
                                              2        296            346   
                                              3        296            349   
                                              4        259            371   
                                              5        264            320   
...                                                    ...            ...   
OT-1     null    10pM          2         72.0 8464     338            621   
                                              8465     268            823   
                                              8466     295            809   
                                              8467     2

In [3]:
group_sizes = [2,4,10,16,32,64,128,256,512]


In [4]:
def avg(group,group_size): 
    group['GroupNumber'] = np.array(range(len(group.index))) // group_size
    res = group.groupby('GroupNumber').mean()
    return res

In [5]:
history_list_acc = []
history_list_val_acc = []
history_list_val_loss = []
history_list_loss = []

Varying Batch Sizes

In [12]:
group_sizes = [10,16,32]
batch_sizes = [128, 256, 512, 1024, 2048]

In [16]:
for group_size in group_sizes:
    for BATCH_SIZE in batch_sizes:

        print(group_size, BATCH_SIZE)

        averaged_df = data.groupby(['Peptide', 'Time']).apply(avg, group_size=group_size)
        averaged_df = pd.DataFrame(StandardScaler().fit_transform(averaged_df), columns=averaged_df.columns, index=averaged_df.index)

        antigen = list(averaged_df.index.get_level_values('Peptide'))
        time = list(averaged_df.index.get_level_values('Time'))

        X = np.array(averaged_df.values)
        y = np.array(list(map(lambda x: ANTIGENS.index(x), antigen)))
        y = to_categorical(y)
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_PROPORTION, random_state=42)

        autoencoder = Sequential([
            layers.Dense(15, activation='elu'),
            layers.Dense(10, activation='softsign'),
            layers.Dense(5, activation='swish'),
            layers.Dense(2, activation='linear', name="Bottleneck"), # The bottleneck. 
            layers.Dense(5, activation='selu'), 
            layers.Dense(len(ANTIGENS), activation='softmax', name="Output") # Predicting the input. 
        ])

        autoencoder.compile(optimizer='adagrad', loss=losses.CategoricalCrossentropy(), metrics=['categorical_accuracy'])

        CHECKPOINT_PATH = "training/train_avg/cp-{epoch:04d}.ckpt"
        checkpoint_dir = os.path.dirname(CHECKPOINT_PATH)

        STEPS_PER_EPOCH = X_train.shape[0] / BATCH_SIZE
        SAVE_PERIOD = 10


        training_callback = ModelCheckpoint(filepath=CHECKPOINT_PATH,
                                            save_weights_only=True,
                                            verbose=1, 
                                            save_freq=int(SAVE_PERIOD * STEPS_PER_EPOCH)
                                        )

        # autoencoder.save_weights(CHECKPOINT_PATH.format(epoch=0))

        tf.random.set_seed(42)

        hist = autoencoder.fit(X_train, y_train,
                        epochs=1000,
                        shuffle=True,
                        validation_data=(X_test, y_test),
                        callbacks=[training_callback],
                        batch_size=BATCH_SIZE,
                        verbose=0)
        
        history_list_acc.append(hist.history['categorical_accuracy'])
        history_list_val_acc.append(hist.history['val_categorical_accuracy'])
        history_list_val_loss.append(hist.history['val_loss'])
        history_list_loss.append(hist.history['loss'])

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

        # Plot the accuracy on the first subplot
        ax1.plot(hist.history['categorical_accuracy'], label='Accuracy')
        ax1.plot(hist.history['val_categorical_accuracy'], label='Validation Accuracy')
        ax1.set_title(f'Model Accuracy for groupsize {group_size} and batch size {BATCH_SIZE}')
        ax1.set_xlabel('Epoch')
        ax1.set_ylabel('Accuracy')
        ax1.legend()

        # Plot the loss on the second subplot
        ax2.plot(hist.history['loss'], label='Loss')
        ax2.plot(hist.history['val_loss'], label='Validation Loss')
        ax2.set_title('Model Loss')
        ax2.set_xlabel('Epoch')
        ax2.set_ylabel('Loss')
        ax2.legend()

        # Adjust the spacing between subplots
        plt.tight_layout()

KeyboardInterrupt: 